In [ ]:
"""
Statistical Significance Testing Script.

Original VS. Original(+1000)
"""

import numpy as np
import pandas as pd
from scipy import stats

DATA = {

    #  5-CLASS TASK (Original VS. Original(+1000))

    ("5-Class", "Accuracy"): {
        "Baseline":  [73.81, 72.86, 70.68, 70.82, 68.34, 72.42, 79.20, 75.73, 75.32],
        "Augmented": [74.63, 73.93, 71.74, 71.67, 69.31, 73.25, 79.24, 75.28, 75.00]
    },
    ("5-Class", "Weighted-F1"): {
        "Baseline":  [73.52, 72.61, 70.33, 70.35, 67.75, 71.96, 79.10, 75.33, 74.95],
        "Augmented": [74.46, 73.70, 71.41, 71.39, 68.94, 72.84, 79.16, 74.97, 74.70]
    },
    ("5-Class", "Macro-F1"): {
        "Baseline":  [57.83, 56.77, 54.26, 54.23, 51.08, 56.57, 72.26, 59.45, 59.93],
        "Augmented": [60.53, 57.87, 55.37, 57.36, 53.80, 57.53, 75.37, 61.31, 60.87]
    },

    #  Binary TASK (Original VS. Original(+1000))
    ("Binary", "Accuracy"): {
        "Baseline":  [92.50, 93.13, 89.74, 89.23, 88.60, 93.04, 96.52, 93.70, 93.04],
        "Augmented": [93.28, 92.80, 90.37, 89.77, 87.85, 92.77, 96.52, 94.03, 93.52]
    },
    ("Binary", "Weighted-F1"): {
        "Baseline":  [92.53, 93.09, 89.63, 89.04, 88.37, 92.88, 96.49, 93.65, 92.95],
        "Augmented": [93.25, 92.72, 90.23, 89.63, 87.42, 92.66, 96.49, 93.99, 93.42]
    },
    ("Binary", "Macro-F1"): {
        "Baseline":  [87.20, 88.03, 81.96, 80.83, 79.62, 87.50, 93.90, 88.99, 87.73],
        "Augmented": [88.33, 87.33, 82.95, 81.93, 77.65, 87.16, 93.90, 89.58, 88.52]
    }
}

def holm_bonferroni(p_values):
    indexed = sorted(enumerate(p_values), key=lambda x: x[1])    #Enforces monotonicity in Holm-Bonferroni step-down correction.
    m = len(p_values)
    adj = [0.0] * m
    prev = 0.0
    for rank, (original_idx, p) in enumerate(indexed):
        val = min(p * (m - rank), 1.0)
        val = max(val, prev)  # enforce monotonicity
        adj[original_idx] = val
        prev = val
    return adj

records = []
raw_wilcox_p = []
raw_ttest_p = []

for (task, metric), values in DATA.items():
    base = np.array(values["Baseline"], dtype=float)
    aug = np.array(values["Augmented"], dtype=float)
    diff = aug - base

    d_nonzero = diff[diff != 0]
    n_nonzero = len(d_nonzero)
    n_total = len(diff)

    mean_base = np.mean(base)
    mean_aug = np.mean(aug)
    mean_gain = np.mean(diff)
    wins = int(np.sum(diff > 0))

    # 1. Shapiro-Wilk test on differences
    shapiro_stat, p_shapiro = stats.shapiro(diff)
    is_normal = "Yes" if p_shapiro >= 0.05 else "No"

    # 2. Statistical Tests
    t_stat, p_ttest = stats.ttest_rel(aug, base)
    w_res = stats.wilcoxon(aug, base)
    p_wilcox = w_res.pvalue

    raw_wilcox_p.append(p_wilcox)
    raw_ttest_p.append(p_ttest)

    # 3. Correct Wilcoxon Z & Effect Size r
    ranks = stats.rankdata(np.abs(d_nonzero))
    w_plus = ranks[d_nonzero > 0].sum()
    w_minus = ranks[d_nonzero < 0].sum()

    mean_w = n_nonzero * (n_nonzero + 1) / 4.0
    std_w = np.sqrt(n_nonzero * (n_nonzero + 1) * (2 * n_nonzero + 1) / 24.0)
    z_val = (w_plus - mean_w) / std_w
    effect_r = abs(z_val) / np.sqrt(n_nonzero)

    # 4. Cohen's d for paired samples
    cohen_d = mean_gain / np.std(diff, ddof=1)

    records.append({
        "Task": task,
        "Metric": metric,
        "Mean_Base": round(mean_base, 2),
        "Mean_Aug": round(mean_aug, 2),
        "Gain": round(mean_gain, 2),
        "Wins": f"{wins}/{n_total}",
        "Shapiro_p": round(p_shapiro, 4),
        "Is_Normal": is_normal,
        "Wilcoxon_p": round(p_wilcox, 4),
        "ttest_p": round(p_ttest, 4),
        "Z_val": round(z_val, 3),
        "Effect_r": round(effect_r, 3),
        "Cohens_d": round(cohen_d, 2)
    })

# Apply Holm to both sets of p-values
adj_p_wilcox = holm_bonferroni(raw_wilcox_p)
adj_p_ttest = holm_bonferroni(raw_ttest_p)

for i, rec in enumerate(records):
    rec["Holm_Wilcoxon_p"] = round(adj_p_wilcox[i], 4)
    rec["Holm_ttest_p"] = round(adj_p_ttest[i], 4)
    rec["Sig_Wilcoxon"] = "Yes" if adj_p_wilcox[i] < 0.05 else "No"
    rec["Sig_ttest"] = "Yes" if adj_p_ttest[i] < 0.05 else "No"

df_final = pd.DataFrame(records)
print(df_final.to_string(index=False))



output_file = "comprehensive_significance_report_OvsO1000.csv"
df_final.to_csv(output_file, index=False)


try:
    from google.colab import files
    files.download(output_file)
except ImportError:
    print(f"Saved locally'{output_file}'")